# LLMs & Generative AI — Session 1 Lab: Foundations & Prompt Engineering
### ITI · Instructor: Ahmed Abdelsalam

Two halves today:
- **Part A (notebook)** — look inside a real model: tokens, next-token probabilities, temperature.
- **Part B (browser)** — a prompt-engineering worksheet you do in **ChatGPT or Claude**, because a 0.5B model is too small to show what good prompting really buys you.

**Rules of the lab**
- Made for **Google Colab**. GPU runtime is faster: *Runtime → Change runtime type → GPU*.
- Try first, ask second.
- Done early? Head to the **Challenges**.

## Setup — run this first

First run downloads two small models (~1.2 GB total). Takes a minute.

In [1]:
!pip install transformers -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# small base model — for looking at tokens and probabilities
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

# small instruct model — for chatting
chat_id = "Qwen/Qwen2.5-0.5B-Instruct"
chat_tok = AutoTokenizer.from_pretrained(chat_id)
chat_model = AutoModelForCausalLM.from_pretrained(chat_id).to(device)

def ask(prompt, max_new_tokens=60, temperature=None):
    """Send a prompt to the chat model and return its reply."""
    messages = [{"role": "user", "content": prompt}]
    text = chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = chat_tok(text, return_tensors="pt").to(device)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=chat_tok.eos_token_id)
    if temperature is None:
        kwargs["do_sample"] = False
    else:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    with torch.no_grad():
        out = chat_model.generate(**inputs, **kwargs)
    return chat_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Ready.")

Using: cpu


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ready.


# Part A — Looking inside the model

## Exercise 1 — Tokenization

1. Tokenize the sentence `"Machine learning models need good data."` and print the token list, the IDs, and how many tokens it is.
2. Then tokenize these four and print the pieces for each: `"dog"`, `"undeniably"`, `"Ismailia"`, `"ChatGPT"`.
3. In a comment, note which ones were split into more than one token and why you think so.

In [2]:
# Exercise 1 - Tokenization

text = "Machine learning models need good data."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Tokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nNumber of tokens:")
print(len(token_ids))

# Tokenize each word
words = ["dog", "undeniably", "Ismailia", "ChatGPT"]

for word in words:
    print(f"\n{word}:")
    print(tokenizer.tokenize(word))


Tokens:
['Machine', 'Ġlearning', 'Ġmodels', 'Ġneed', 'Ġgood', 'Ġdata', '.']

Token IDs:
[37573, 4673, 4981, 761, 922, 1366, 13]

Number of tokens:
7

dog:
['dog']

undeniably:
['und', 'en', 'iably']

Ismailia:
['Is', 'mail', 'ia']

ChatGPT:
['Chat', 'G', 'PT']


## Exercise 2 — Next-token probabilities

Write a function `next_tokens(prompt, k=5)` that:
1. Tokenizes the prompt.
2. Runs the model and takes the logits for the **last** position.
3. Applies softmax and prints the top-k tokens with their percentages.

Test it on `"The best thing about machine learning is"` and on `"2 + 2 ="`.

In [3]:
# Your code here

import torch

def next_tokens(prompt, k=5):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k)

    print(f"\nPrompt: {prompt}")
    print("-" * 50)

    for prob, idx in zip(top_probs, top_ids):
        token = tokenizer.decode([idx])
        print(f"Token: {repr(token):<15} Probability: {prob.item()*100:.2f}%")
next_tokens("The best thing about machine learning is")
next_tokens("2 + 2 =")


Prompt: The best thing about machine learning is
--------------------------------------------------
Token: ' that'         Probability: 64.48%
Token: ' it'           Probability: 10.06%
Token: ' the'          Probability: 5.41%
Token: ' how'          Probability: 2.91%
Token: ' you'          Probability: 1.87%

Prompt: 2 + 2 =
--------------------------------------------------
Token: ' 3'            Probability: 10.19%
Token: ' 1'            Probability: 9.16%
Token: ' 2'            Probability: 8.42%
Token: ' 0'            Probability: 7.52%
Token: ' 4'            Probability: 5.50%


## Exercise 3 — Greedy generation loop

Write a loop that generates **15 tokens** starting from `"In the future, computers will"`, always choosing the most likely token (`argmax`). Print the growing text.

*This is the whole generation mechanism written out by hand.*

In [4]:
# Your code here

prompt = "In the future, computers will"

text = prompt

for i in range(15):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
    next_token_id = outputs.logits[:, -1].argmax(dim=-1)
    next_token = tokenizer.decode(next_token_id)

    text += next_token
    print(f"Step {i+1}:")
    print(text)

Step 1:
In the future, computers will be
Step 2:
In the future, computers will be able
Step 3:
In the future, computers will be able to
Step 4:
In the future, computers will be able to do
Step 5:
In the future, computers will be able to do things
Step 6:
In the future, computers will be able to do things like
Step 7:
In the future, computers will be able to do things like send
Step 8:
In the future, computers will be able to do things like send and
Step 9:
In the future, computers will be able to do things like send and receive
Step 10:
In the future, computers will be able to do things like send and receive messages
Step 11:
In the future, computers will be able to do things like send and receive messages,
Step 12:
In the future, computers will be able to do things like send and receive messages, and
Step 13:
In the future, computers will be able to do things like send and receive messages, and they
Step 14:
In the future, computers will be able to do things like send and receive mess

## Exercise 4 — Temperature experiments

Using the `ask()` function from Setup, run the prompt
`"Give me one creative name for a coffee shop."`
**three times each** at temperature `0.2` and at `1.5`.

Print them grouped, then write a one-line comment on the difference you observe.

In [6]:
# Exercise 4 - Temperature experiments

prompt = "Give me one creative name for a coffee shop."

print("Temperature = 0.2")


for i in range(3):
    print(f"Run {i+1}:")
    print(ask(prompt, temperature=0.2))
    print()


print("Temperature = 1.5")


for i in range(3):
    print(f"Run {i+1}:")
    print(ask(prompt, temperature=1.5))
    print()


Temperature = 0.2
Run 1:
"Bean and Brew: A Coffee Haven"

Run 2:
"Bean & Brew: A Coffee Haven"

Run 3:
"Bean Bliss Brew Haven"

Temperature = 1.5
Run 1:
Certainly! What is the unique charm or character your preferred concept should have? It could be something as simple as "Beans to Sip", a fusion of "beans" and "spice" to emphasize the essence of a casual café, while maintaining the cozy ambiance. If you want it to blend

Run 2:
"Whispers in a Glass Bottle" -- "Coffee Room" or "Sip and Talk Bar". Choose a unique blend of words from the sentence for your coffee shop's name. "Tartlet Tea Haven" might suggest a place where you can find delicious green teas and their herbal tea blends

Run 3:
Certainly! A unique and intriguing name could be something like "Mint Haven". This suggests that it has some flavor of mint or herb but could have an unusual, unexpected name due to its somewhat surreal setting. Additionally, it emphasizes the concept of tranquility and connection to nature in a way t

## Exercise 5 — Zero-shot vs few-shot

We want the model to classify sentiment as exactly `positive` or `negative`.

1. **Zero-shot:** ask it to classify `"The delivery was late and the box was damaged."` with no examples.
2. **Few-shot:** ask again, but first give it two worked examples in the prompt.

Print both answers and compare how well each followed the format.

In [7]:
# Your code here
review = "The delivery was late and the box was damaged."

zero_shot_prompt = f"""
Classify the sentiment of the following review.
Answer with exactly one word: positive or negative.

Review:
"{review}"
"""
few_shot_prompt = f"""
Classify the sentiment as exactly positive or negative.

Example 1:
Review: "The product is amazing and arrived on time."
Answer: positive

Example 2:
Review: "The phone stopped working after one day."
Answer: negative

Now classify this review.

Review:
"{review}"

Answer:
"""

print("Zero-shot:")
print(ask(zero_shot_prompt))


print("Few-shot:")
print(ask(few_shot_prompt))


Zero-shot:
negative
Few-shot:
Answer: negative


# Part B — Prompt engineering worksheet (in the browser)

Open **ChatGPT** or **Claude** in a browser tab. The model in this notebook is only 0.5B parameters — good for showing mechanics, too small to show what careful prompting really buys you.

For each task below: run the **weak** prompt, run the **strong** prompt, then record what changed. Fill in the answer cells as text.

### Task 1 — Explaining a concept

**Weak prompt:**
```
explain gradient descent
```

**Strong prompt:**
```
You are a patient tutor for a student who has just trained their first
neural network. Explain gradient descent in exactly 4 short bullet points.
Use a real-world analogy, no mathematics, and keep it under 120 words.
```

Run both. In the cell below, write 2–3 sentences on what changed.

In [ ]:
# Your code here
The weak prompt returned a general explanation with no clear format. The strong prompt was shorter, well organized, and matched all the requested instructions. It was easier to understand because it used a real-world analogy and simple language.

### Task 2 — Structured output

Ask the model to extract information from this text:

```
Ahmed Hassan, 24, is a machine learning engineer at a startup in Cairo.
He studied Computer Science at Suez Canal University and works mainly on
computer vision projects.
```

**Weak prompt:** `get the details from this text`

**Strong prompt:** ask for **valid JSON only**, with the exact keys
`name`, `age`, `job`, `city`, `university`, and no extra commentary.

Paste the strong prompt's output below and note whether the JSON was valid.

In [ ]:
# Your code here
The weak prompt gave the information in plain text. The strong prompt returned a valid JSON with the required keys only, making the output more organized and machine-readable.

### Task 3 — Role and audience

Pick any topic you know well (YOLO, CNNs, transfer learning...).

Ask for an explanation **three times**, changing only the audience:
1. "Explain <topic> to a 10-year-old."
2. "Explain <topic> to a first-year CS student."
3. "Explain <topic> to a hiring manager deciding whether to fund the project."

Note in the cell below what changed between the three.

In [ ]:
# Your code here
The content was adapted to match each audience. The explanation for a child was simple, the one for a CS student was more technical, and the one for a hiring manager emphasized practical impact and business value.

### Task 4 — Your own comparison

Choose a task you'd actually use an LLM for. Write a deliberately weak prompt and a strong one applying **at least three** of the five techniques (be specific · give context · assign a role · show examples · fix the output format).

Record both prompts and which techniques you used.

In [ ]:
# Your code here
The weak prompt was vague, so the response was general and less useful. The strong prompt gave clear context, assigned a role, and specified the required output format, which made the answer more accurate, organized, and relevant to the task.

## 🚀 Challenges

Add new cells as needed.

**Challenge A — Token cost of Arabic.**
Tokenize the English sentence `"Artificial intelligence is changing the world"` and its Arabic translation `"الذكاء الاصطناعي يغير العالم"`. Print the token count of each. Which costs more tokens, and why does that matter for API pricing?

In [10]:
english = "Artificial intelligence is changing the world"
arabic = "الذكاء الاصطناعي يغير العالم"

eng_tokens = tokenizer.encode(english)
ar_tokens = tokenizer.encode(arabic)

print("English token count:", len(eng_tokens))
print("Arabic token count:", len(ar_tokens))


English token count: 7
Arabic token count: 26


**Challenge B — Confidence and correctness.**
Use `next_tokens()` on `"The largest planet in our solar system is"`. Is the correct answer the top prediction? What does its probability tell you about how *sure* the model is?

In [11]:
next_tokens("The largest planet in our solar system is")


Prompt: The largest planet in our solar system is
--------------------------------------------------
Token: ' about'        Probability: 9.38%
Token: ' the'          Probability: 6.21%
Token: ' a'            Probability: 5.83%
Token: ' Earth'        Probability: 2.82%
Token: ' Jupiter'      Probability: 2.70%


**Challenge C — Prompt that fails.**
Find a prompt where the small `ask()` model gives a clearly wrong or nonsensical answer. Print the prompt and its answer, and write one line on why you think it failed.

In [13]:
prompt = "What is the capital of Egypt? Answer with one word only."

answer = ask(prompt)

print("Prompt:")
print(prompt)

print("\nAnswer:")
print(answer)

Prompt:
What is the capital of Egypt? Answer with one word only.

Answer:
Cairo
